# MAMA-MIA: tải từ XNAT + tiền xử lý Step 1-3 trên Colab
Chạy trên **CPU runtime** (không cần GPU). Raw NIfTI nằm ở ổ cục bộ `/content/tmp` và bị xóa sau mỗi ca; chỉ `.npz` (~3.5 GB) lưu lên Drive.
Trước khi chạy: upload cả thư mục project (có `src/`, `scripts/`, `configs/`) lên Drive tại `MyDrive/CS406-SCB-TumorSeg`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/CS406-SCB-TumorSeg'
%cd $PROJ

In [ ]:
!pip install -q SimpleITK scikit-image scikit-learn pandas openpyxl pyyaml tqdm requests

In [ ]:
import os, re
os.makedirs('data_real/raw', exist_ok=True)
# Bảng lâm sàng thật (public, không cần đăng nhập)
!curl -L --fail -o data_real/raw/clinical_and_imaging_info.xlsx https://xnat.health-ri.nl/data/projects/mama_mia/resources/2286069/files/clinical_and_imaging_info.xlsx
# config: mọi đường dẫn data_real/* trỏ vào Drive
cfg = open('configs/default_config.yaml', encoding='utf-8').read()
cfg = re.sub(r'"data/(raw|processed|splits)', r'"data_real/\1', cfg)
open('configs/real_config.yaml', 'w', encoding='utf-8').write(cfg)

Thử 3 ca trước để đo tốc độ, sau đó chạy toàn bộ.

In [ ]:
!python scripts/download_process_xnat.py --config configs/real_config.yaml --workers 4 --limit 3 --tmp-dir /content/tmp

In [ ]:
# Toàn bộ 1480 ca. Resumable: nếu Colab ngắt kết nối, chạy lại đúng cell này.
!python scripts/download_process_xnat.py --config configs/real_config.yaml --workers 12 --tmp-dir /content/tmp

In [ ]:
import glob
print(len(glob.glob('data_real/processed/npz_96x96x96/*.npz')), '/ 1480 npz')
!cat outputs/logs/xnat_failed_cases.tsv 2>/dev/null | head